In [2]:
# =============================================
# TASK 1: News Topic Classifier with BERT - FIXED for latest Transformers
# =============================================

# Step 1: Install
!pip install -q transformers datasets evaluate accelerate gradio

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import evaluate
import gradio as gr

# Step 2: Load AG News dataset
print("Loading the AG News dataset...")
dataset = load_dataset("ag_news", split="train[:10000]")   # small fast subset
dataset = dataset.train_test_split(test_size=0.2)
print(f"Dataset loaded! Train samples: {len(dataset['train'])}")

# Step 3: Tokenize
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

# Step 4: Load model (4 classes: World, Sports, Business, Sci/Tech)
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=4)

# Step 5: Training setup - FIXED HERE
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",           # ← This was the problem (now eval_strategy)
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    report_to="none",
    logging_steps=10,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy"
)

# Step 6: Metrics
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(-1)
    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "f1": f1.compute(predictions=predictions, references=labels, average="weighted")["f1"]
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

# Step 7: Train! (This will take 5-12 minutes on Colab GPU)
print("🚀 Starting training...")
trainer.train()

# Step 8: Final evaluation
results = trainer.evaluate()
print("\n✅ Final Results:")
print(f"Accuracy : {results['eval_accuracy']:.4f}")
print(f"F1-Score : {results['eval_f1']:.4f}")

# Step 9: Live Gradio Demo
def classify_news(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]
    labels = ["World", "Sports", "Business", "Sci/Tech"]
    return {labels[i]: float(probs[i]) for i in range(4)}

demo = gr.Interface(
    fn=classify_news,
    inputs=gr.Textbox(label="Paste any news headline here", placeholder="Example: Arsenal wins Premier League title..."),
    outputs=gr.Label(label="Predicted Topic (with confidence)"),
    title="📰 News Topic Classifier (BERT) - DevelopersHub Internship",
    description="Fine-tuned by Subhan | Try it live!",
    examples=[
        ["Pakistan wins cricket world cup final against India"],
        ["Apple releases new iPhone with revolutionary AI features"],
        ["Stock market crashes after new economic data"],
        ["Scientists discover new planet in habitable zone"]
    ]
)

demo.launch(share=True)   # ← Click the public URL to test!

Loading the AG News dataset...
Dataset loaded! Train samples: 8000


Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.238164,0.262222,0.918000,0.918070
2,0.208820,0.259386,0.920500,0.920252


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


✅ Final Results:
Accuracy : 0.9205
F1-Score : 0.9203
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d3bb5f06682afd1c69.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
